In [132]:
import numpy as np
import pandas as pd

In [142]:
df = pd.read_csv('/home/tpss/Music/ml/song/songdata.csv.zip');
df.head(3)

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \nAnd..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \nTouch me gentl..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \nWhy I had t...


In [143]:
df = df.sample(n= 1000).drop('link', axis = 1).reset_index(drop= True)

In [144]:
df.head(5)

,artist,song,text
0,Kirsty Maccoll,Keep Your Hands Off My Baby,We've been friends for oh so long \nI let you...
1,Whitesnake,Cheap An' Nasty,"Come on baby \n \nI get so confused, \nBut,..."
2,Black Sabbath,Time Machine,Oh what are you gonna do \nWhen there's a par...
3,Leann Rimes,White Christmas,I'm dreaming of a white Christmas \nJust like...
4,Iggy Pop,I Felt The Luxury,She sat on the pavement \nAs I pulled in the ...


In [145]:
df['text'] =  df['text'].str.lower().replace(r'W\S', '').replace('\\n', ' ', regex = True)

In [146]:
df['text']

0      we've been friends for oh so long   i let you ...
1      come on baby      i get so confused,   but, my...
2      oh what are you gonna do   when there's a part...
3      i'm dreaming of a white christmas   just like ...
4      she sat on the pavement   as i pulled in the d...
                             ...                        
995    i will give you life   you'll serve only me   ...
996    i think this time you've said enough   to make...
997    oh, he makes his life as a carpenter   he work...
998    on we plow   the big bully try to stick his fi...
999    hang up the phone, i'm alone   i'll unlock the...
Name: text, Length: 1000, dtype: object

In [147]:
import nltk 
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize

stemmer = PorterStemmer()


def Tokenizer(text):
    tokens = word_tokenize(text)  # Tokenize the text
    stemmed_words = [stemmer.stem(t) for t in tokens]  # Corrected to use `t`
    return " ".join(stemmed_words)
    

In [148]:
df['text'] =  df['text'].apply(lambda x:Tokenizer(x) )

In [149]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tfid = TfidfVectorizer(stop_words = 'english')
matrix =  tfid.fit_transform(df['text'])
matrix.shape
similarity = cosine_similarity(matrix)

In [150]:
df['song'][0]

'Keep Your Hands Off My Baby'

In [151]:
df[df['song'] == "Don't Know How (Not To Love You)"]

,artist,song,text


In [152]:
def Recommendation(song):
    try:
        # Check if the song exists in the DataFrame
        if song not in df['song'].values:
            return f"Error: The song '{song}' does not exist in the dataset."
        
        # Find the index of the song
        idx = df[df['song'] == song].index[0]
        
        # Calculate distances and sort them
        distances = sorted(enumerate(similarity[idx]), reverse=False, key=lambda x: x[1])
        
        # Collect the top 20 similar songs
        songs = []
        for i in distances[1:21]:  # Skip the first element since it is the song itself
            songs.append(df.iloc[i[0]]['song'])
        
        return songs
    except Exception as e:
        return f"An error occurred: {e}"


In [154]:
# Recomendation("Don't Know How (Not To Love You)")

In [156]:
import pickle
pickle.dump(similarity,   open('similarity.pkl', 'wb'))
pickle.dump(df,   open('df.pkl', 'wb'))


